# 06 — Final Evaluation, Error Analysis and Model Export

This notebook:

1. Loads the three tuned candidate pipelines from Notebook 05.
2. Evaluates them on the same held-out test split.
3. Selects the best model by macro F1.
4. Produces a classification report and confusion matrix.
5. Saves wrong predictions and error summaries.
6. Saves the selected final pipeline.
7. Demonstrates inference on new review text.

Because the vectorizer is inside the scikit-learn `Pipeline`, one `.pkl` file contains both the text vectorizer and classifier.

In [ ]:
from pathlib import Path
import shutil
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

PROJECT_ROOT = Path("..")
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

test_df = pd.read_csv(RESULTS_DIR / "test_split.csv")

X_test = test_df["processed_text"].astype(str)
y_test = test_df["sentiment"].astype(str)

candidate_paths = {
    "Naive Bayes Tuned":
        MODELS_DIR / "naive_bayes_tuned_pipeline.pkl",
    "Logistic Regression Tuned":
        MODELS_DIR / "logistic_regression_tuned_pipeline.pkl",
    "Support Vector Machine Tuned":
        MODELS_DIR / "svm_tuned_pipeline.pkl"
}

print("Test rows:", len(test_df))
print(y_test.value_counts())

## Evaluate all tuned candidates

In [ ]:
evaluation_rows = []
candidate_predictions = {}
candidate_models = {}

for model_name, path in candidate_paths.items():
    model = joblib.load(path)
    pred = model.predict(X_test)

    candidate_models[model_name] = model
    candidate_predictions[model_name] = pred

    evaluation_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro Precision": precision_score(
            y_test, pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_test, pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_test, pred,
            average="macro",
            zero_division=0
        )
    })

final_evaluation_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

display(final_evaluation_df)

## Select best tuned model

In [ ]:
best_model_name = final_evaluation_df.iloc[0]["Model"]
best_pipeline = candidate_models[best_model_name]
best_pred = candidate_predictions[best_model_name]

print("Selected best model:", best_model_name)
print(
    "Macro F1:",
    final_evaluation_df.iloc[0]["Macro F1"]
)

## Classification report

In [ ]:
report_dict = classification_report(
    y_test,
    best_pred,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T
display(report_df)

print(
    classification_report(
        y_test,
        best_pred,
        zero_division=0
    )
)

## Confusion matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    best_pred,
    labels=best_pipeline.classes_
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=best_pipeline.classes_
)

disp.plot(values_format="d")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()

figure_path = FIGURES_DIR / "best_model_confusion_matrix.png"
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved figure:", figure_path)

## Wrong prediction analysis

In [ ]:
wrong_prediction_df = pd.DataFrame({
    "processed_text": X_test.reset_index(drop=True),
    "actual_sentiment": y_test.reset_index(drop=True),
    "predicted_sentiment": best_pred
})

wrong_prediction_df = wrong_prediction_df[
    wrong_prediction_df["actual_sentiment"]
    != wrong_prediction_df["predicted_sentiment"]
].copy()

display(wrong_prediction_df)

wrong_prediction_summary = (
    wrong_prediction_df
    .groupby(
        ["actual_sentiment", "predicted_sentiment"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(wrong_prediction_summary)

## Save evaluation outputs

In [ ]:
final_evaluation_df.to_csv(
    RESULTS_DIR / "final_model_evaluation.csv",
    index=False
)

report_df.to_csv(
    RESULTS_DIR / "best_model_classification_report.csv"
)

wrong_prediction_df.to_csv(
    RESULTS_DIR / "error_analysis.csv",
    index=False
)

wrong_prediction_summary.to_csv(
    RESULTS_DIR / "wrong_prediction_summary.csv",
    index=False
)

print("Saved evaluation and error-analysis files.")

## Save selected final pipeline

In [ ]:
final_model_path = MODELS_DIR / "best_sentiment_model_pipeline.pkl"

joblib.dump(
    best_pipeline,
    final_model_path
)

print("Saved:", final_model_path)

## Reload and test on new reviews

In [ ]:
loaded_pipeline = joblib.load(
    MODELS_DIR / "best_sentiment_model_pipeline.pkl"
)

new_reviews = [
    "good benefits but management is stressful and workload is too heavy",
    "toxic workplace and no career growth",
    "salary low"
]

predictions = loaded_pipeline.predict(new_reviews)

for review, prediction in zip(new_reviews, predictions):
    print(f"{review} -> {prediction}")